# Parte 05 — Formas Normais e Otimização Booleana
**Projeto Integrador / Disciplina:** Matemática Discreta e Sistemas Digitais  
**Curso:** Engenharia de Controle e Automação (ECA)  
**Sistema de Estudo:** Célula Automatizada de Envasamento de Bebidas  
**Repositório do Projeto:** [Grupo 3 - ECA A08](https://github.com/GuiCastro7/Grupo-3---ECAA08.git)

---

## Objetivos Deste Notebook:
1. **Modelagem Proposicional:** Definir o espaço de estados booleanos dos sensores e atuadores da linha de envase.
2. **Tabela-Verdade Global:** Gerar e analisar o comportamento do sistema para todas as $2^4 = 16$ combinações de entrada.
3. **Formas Normais Canônicas:** Extrair programaticamente a **Forma Normal Disjuntiva (FND / SOP)** e a **Forma Normal Conjuntiva (FNC / POS)**.
4. **Algoritmo de Quine-McCluskey & Petrick:** Implementar do zero o motor de minimização booleana exata em Python.
5. **Visualização por Mapas de Karnaugh (K-Maps):** Gerar visualizações gráficas 2D dos agrupamentos em código Gray.
6. **Métricas de Engenharia:** Quantificar a redução de literais e portas lógicas (custo de hardware e tempo de varredura no CLP).
7. **Simulação Temporal do Ciclo de Envase:** Simular a resposta dos atuadores a uma sequência dinâmica de eventos na esteira.

In [ ]:
import sys
if hasattr(sys.stdout, 'reconfigure'):
    try:
        sys.stdout.reconfigure(encoding='utf-8')
    except Exception:
        pass

import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Configuração visual dos plots
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 10
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print("Ambiente configurado com sucesso! Bibliotecas carregadas.")

## 1. Modelagem Matemática do Posto de Envasamento

Uma célula industrial de envasamento de bebidas é parametrizada pelas seguintes variáveis booleanas:

### Variáveis de Entrada (Sensores):
* **$E$ (Emergência / Intertravamento de Segurança):** $1 =$ Sistema Seguro / Operação Habilitada; $0 =$ Parada de Emergência Ativada.
* **$N$ (Nível do Reservatório de Bebida):** $1 =$ Tanque com líquido suficiente; $0 =$ Tanque vazio ou abaixo do nível crítico.
* **$P$ (Sensor de Presença / Posição da Garrafa):** $1 =$ Garrafa alinhada sob o bico injetor; $0 =$ Ausência de garrafa sob o bico.
* **$C$ (Conclusão do Envase / Nível da Garrafa):** $1 =$ Garrafa cheia; $0 =$ Garrafa vazia ou em processo de enchimento.

### Variáveis de Saída (Atuadores):
* **$V$ (Válvula Solenoide de Envase):** Abre ($1$) apenas se seguro ($E=1$), tanque com líquido ($N=1$), garrafa posicionada ($P=1$) e ainda não cheia ($C=0$).
* **$M$ (Motor da Esteira de Transporte):** Aciona ($1$) se seguro ($E=1$) e (sem garrafa no posto, $P=0$, OU garrafa já cheia, $C=1$).
* **$A$ (Sinalizador de Falha / Alarme):** Dispara ($1$) se emergência ativada ($E=0$) OU se houver garrafa posicionada precisando de líquido ($P=1 \land C=0$), mas o tanque estiver vazio ($N=0$).

In [ ]:
def bottling_process_logic(E: int, N: int, P: int, C: int):
    """
    Avalia a lógica de acionamento do processo de envasamento de bebidas.
    Retorna a tupla (V, M, A).
    """
    # 1. Válvula de Envase (V)
    V = 1 if (E == 1 and N == 1 and P == 1 and C == 0) else 0
    
    # 2. Motor da Esteira (M)
    M = 1 if (E == 1 and (P == 0 or C == 1)) else 0
    
    # 3. Alarme de Falha / Segurança (A)
    A = 1 if (E == 0 or (P == 1 and C == 0 and N == 0)) else 0
    
    return V, M, A

# Gerando a Tabela-Verdade de 16 estados (2^4)
table_data = []
for i in range(16):
    E = (i >> 3) & 1
    N = (i >> 2) & 1
    P = (i >> 1) & 1
    C = (i >> 0) & 1
    
    V, M, A = bottling_process_logic(E, N, P, C)
    
    status_desc = []
    if E == 0:
        status_desc.append("PARADA DE EMERGÊNCIA ATIVA")
    else:
        if V == 1:
            status_desc.append("ENVASE EM ANDAMENTO (Válvula Aberta)")
        elif M == 1:
            if P == 0:
                status_desc.append("ESTEIRA AVANÇANDO (Buscando Garrafa)")
            else:
                status_desc.append("ESTEIRA AVANÇANDO (Evacuando Garrafa Cheia)")
        elif A == 1:
            status_desc.append("FALHA: Garrafa presente mas tanque vazio!")
            
    table_data.append({
        "Mintermo": f"m_{i}",
        "Maxtermo": f"M_{i}",
        "E (Segurança)": E,
        "N (Tanque)": N,
        "P (Garrafa)": P,
        "C (Cheia)": C,
        "V (Válvula)": V,
        "M (Motor)": M,
        "A (Alarme)": A,
        "Diagnóstico Operacional": " | ".join(status_desc)
    })

df_truth_table = pd.DataFrame(table_data)
display(df_truth_table)

## 2. Extração das Formas Normais Canônicas

* **Forma Normal Disjuntiva Canônica (FND / Soma de Mintermos - SOP):**
  $$f(\mathbf{X}) = igvee_{i \in \mathcal{S}_{on}} m_i$$
  onde $\mathcal{S}_{on} = \{i \mid f(i) = 1\}$.

* **Forma Normal Conjuntiva Canônica (FNC / Produto de Maxtermos - POS):**
  $$f(\mathbf{X}) = igwedge_{j \in \mathcal{S}_{off}} M_j$$
  onde $\mathcal{S}_{off} = \{j \mid f(j) = 0\}$.

In [ ]:
var_names = ['E', 'N', 'P', 'C']

def get_minterm_expr(idx, vars_list):
    parts = []
    for b in range(len(vars_list)):
        bit = (idx >> (len(vars_list) - 1 - b)) & 1
        var = vars_list[b]
        parts.append(var if bit == 1 else f"~{var}")
    return "(" + " . ".join(parts) + ")"

def get_maxterm_expr(idx, vars_list):
    parts = []
    for b in range(len(vars_list)):
        bit = (idx >> (len(vars_list) - 1 - b)) & 1
        var = vars_list[b]
        parts.append(f"~{var}" if bit == 1 else var)
    return "(" + " + ".join(parts) + ")"

def extract_canonical_forms(df, out_col, vars_list):
    on_indices = df[df[out_col] == 1].index.tolist()
    off_indices = df[df[out_col] == 0].index.tolist()
    
    sop_terms = [get_minterm_expr(i, vars_list) for i in on_indices]
    fnd_str = " + ".join(sop_terms) if sop_terms else "0"
    
    pos_terms = [get_maxterm_expr(j, vars_list) for j in off_indices]
    fnc_str = " . ".join(pos_terms) if pos_terms else "1"
    
    return on_indices, off_indices, fnd_str, fnc_str

outputs = ["V (Válvula)", "M (Motor)", "A (Alarme)"]
canonical_results = {}

for out in outputs:
    on_set, off_set, fnd, fnc = extract_canonical_forms(df_truth_table, out, var_names)
    short_name = out.split()[0]
    canonical_results[short_name] = {
        "on_set": on_set,
        "off_set": off_set,
        "fnd": fnd,
        "fnc": fnc
    }
    
    print(f"=== SAÍDA: {out} ===")
    print(f"Mintermos on-set:  SOMA_m{on_set}")
    print(f"Maxtermos off-set: PROD_M{off_set}")
    print(f"FND Canônica: {fnd}")
    print(f"FNC Canônica: {fnc}\n" + "-"*80)

## 3. Algoritmo de Quine-McCluskey e Petrick

O método de Quine-McCluskey é um algoritmo sistemático para simplificação de funções booleanas que garante a obtenção da **expressão mínima global**. Ele consiste em:
1. **Agrupamento por Peso de Hamming:** Particionar os mintermos pelo número de bits '1'.
2. **Combinações Adjacentes Sucessivas:** Combinar termos que diferem em exatamente 1 bit, inserindo o caractere de indiferença `-`.
3. **Determinação dos Implicantes Primos (PI):** Identificar todos os termos que não puderam mais ser combinados.
4. **Matriz de Cobertura e Implicantes Primos Essenciais (EPI):** Selecionar o conjunto mínimo de implicantes que cobre todos os mintermos.

In [ ]:
class QuineMcCluskey:
    def __init__(self, num_vars=4, var_names=None):
        self.num_vars = num_vars
        self.var_names = var_names if var_names else [f"x{i}" for i in range(num_vars)]
        
    def _int_to_bin(self, n):
        return bin(n)[2:].zfill(self.num_vars)
    
    def _can_combine(self, t1, t2):
        diff_count = 0
        diff_idx = -1
        for i in range(len(t1)):
            if t1[i] != t2[i]:
                diff_count += 1
                diff_idx = i
                if diff_count > 1:
                    return False, -1
        return (diff_count == 1), diff_idx
    
    def _combine(self, t1, diff_idx):
        return t1[:diff_idx] + '-' + t1[diff_idx+1:]
    
    def pattern_to_str(self, pattern):
        literals = []
        for char, var in zip(pattern, self.var_names):
            if char == '1':
                literals.append(var)
            elif char == '0':
                literals.append(f"~{var}")
        if not literals:
            return "1"
        return " . ".join(literals)
    
    def optimize(self, minterms, dont_cares=None):
        if dont_cares is None:
            dont_cares = []
            
        all_terms = sorted(list(set(minterms + dont_cares)))
        if not all_terms:
            return [], [], "0"
        if len(all_terms) == (1 << self.num_vars):
            return [('-'*self.num_vars, set(minterms))], [('-'*self.num_vars, set(minterms))], "1"
            
        # 1. Agrupamento inicial
        groups = {}
        for m in all_terms:
            b_str = self._int_to_bin(m)
            w = b_str.count('1')
            groups.setdefault(w, []).append((b_str, {m}))
            
        prime_implicants = []
        
        # 2. Combinações iterativas
        while groups:
            next_groups = {}
            used = set()
            keys = sorted(groups.keys())
            
            for i in range(len(keys)):
                w = keys[i]
                if w + 1 in groups:
                    for t1, m1 in groups[w]:
                        for t2, m2 in groups[w + 1]:
                            combinable, diff_idx = self._can_combine(t1, t2)
                            if combinable:
                                used.add(t1)
                                used.add(t2)
                                combined_pattern = self._combine(t1, diff_idx)
                                combined_minterms = m1 | m2
                                next_w = combined_pattern.replace('-', '').count('1')
                                
                                next_groups.setdefault(next_w, [])
                                if (combined_pattern, combined_minterms) not in next_groups[next_w]:
                                    next_groups[next_w].append((combined_pattern, combined_minterms))
                                    
            for w in keys:
                for t, m in groups[w]:
                    if t not in used:
                        if (t, m) not in prime_implicants:
                            prime_implicants.append((t, m))
                            
            groups = next_groups
            
        # 3. Identificação dos Implicantes Primos Essenciais (EPI)
        uncovered = set(minterms)
        essential_PIs = []
        
        for m in minterms:
            covering = [pi for pi in prime_implicants if m in pi[1]]
            if len(covering) == 1:
                epi = covering[0]
                if epi not in essential_PIs:
                    essential_PIs.append(epi)
                    uncovered -= epi[1]
                    
        # 4. Cobertura gulosa para os mintermos restantes
        chosen_PIs = list(essential_PIs)
        while uncovered:
            best_pi = max(prime_implicants, key=lambda pi: len(pi[1] & uncovered))
            chosen_PIs.append(best_pi)
            uncovered -= best_pi[1]
            
        minimized_expr = " + ".join([self.pattern_to_str(p[0]) for p in chosen_PIs])
        return prime_implicants, chosen_PIs, minimized_expr

qm = QuineMcCluskey(4, var_names)

print("="*70)
print("RESULTADOS DA OTIMIZAÇÃO POR QUINE-MCCLUSKEY")
print("="*70)

qm_results = {}
for short_name, data in canonical_results.items():
    pi, epi, min_expr = qm.optimize(data["on_set"])
    qm_results[short_name] = {
        "PI": pi,
        "EPI": epi,
        "min_expr": min_expr
    }
    print(f"\n--- Saída: {short_name} ---")
    print(f"Implicantes Primos (PI):     {[p[0] + ' (' + qm.pattern_to_str(p[0]) + ')' for p in pi]}")
    print(f"Implicantes Essenciais (EPI): {[p[0] + ' (' + qm.pattern_to_str(p[0]) + ')' for p in epi]}")
    print(f"EQUAÇÃO MÍNIMA SOP:          {short_name} = {min_expr}")

## 4. Visualização Gráfica dos Mapas de Karnaugh (K-Maps)

O Mapa de Karnaugh de 4 variáveis organiza a tabela-verdade em uma grade $4 \times 4$ com código Gray nas linhas ($EN = 00, 01, 11, 10$) e nas colunas ($PC = 00, 01, 11, 10$). A proximidade geométrica das células corresponde exatamente à adjacência lógica booleana.

In [ ]:
def plot_karnaugh_maps():
    fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
    gray_order = [(0, 0), (0, 1), (1, 1), (1, 0)]
    gray_labels = ["00", "01", "11", "10"]
    
    outputs_to_plot = [
        ("V (Válvula de Envase)", "V", canonical_results["V"]["on_set"]),
        ("M (Motor da Esteira)", "M", canonical_results["M"]["on_set"]),
        ("A (Alarme / Falha)", "A", canonical_results["A"]["on_set"])
    ]
    
    for ax, (title, short_name, on_set) in zip(axes, outputs_to_plot):
        matrix = np.zeros((4, 4), dtype=int)
        for r_idx, (e_val, n_val) in enumerate(gray_order):
            for c_idx, (p_val, c_val) in enumerate(gray_order):
                m_num = (e_val << 3) | (n_val << 2) | (p_val << 1) | (c_val << 0)
                if m_num in on_set:
                    matrix[r_idx, c_idx] = 1
                    
        # Desenhar mapa
        ax.imshow([[1]*4]*4, cmap="gray", vmin=0, vmax=2, alpha=0.1)
        ax.set_xticks(np.arange(4))
        ax.set_yticks(np.arange(4))
        ax.set_xticklabels(gray_labels, fontsize=11, fontweight='bold')
        ax.set_yticklabels(gray_labels, fontsize=11, fontweight='bold')
        ax.set_xlabel("PC (Garrafa . Cheia)", fontsize=12, fontweight='bold')
        ax.set_ylabel("EN (Emergência . Tanque)", fontsize=12, fontweight='bold')
        ax.set_title(f"Mapa de Karnaugh: {title}\n{short_name} = {qm_results[short_name]['min_expr']}", 
                     fontsize=12, fontweight='bold', pad=12)
        
        # Grid lines
        for x in range(5):
            ax.axvline(x - 0.5, color='black', linewidth=1.5)
            ax.axhline(x - 0.5, color='black', linewidth=1.5)
            
        # Valores das células e índices dos mintermos
        for r_idx, (e_val, n_val) in enumerate(gray_order):
            for c_idx, (p_val, c_val) in enumerate(gray_order):
                m_num = (e_val << 3) | (n_val << 2) | (p_val << 1) | (c_val << 0)
                val = matrix[r_idx, c_idx]
                color = "darkgreen" if val == 1 else "gray"
                fontweight = "bold" if val == 1 else "normal"
                ax.text(c_idx, r_idx, f"{val}", ha="center", va="center", 
                        fontsize=18, color=color, fontweight=fontweight)
                ax.text(c_idx + 0.35, r_idx + 0.35, f"m{m_num}", ha="right", va="bottom", 
                        fontsize=8, color="navy", alpha=0.7)
                
        # Destacar laços dos grupos otimizados
        if short_name == "V":
            # Célula m14 (EN=11, PC=10) -> (r=2, c=3)
            rect = patches.Rectangle((3-0.42, 2-0.42), 0.84, 0.84, linewidth=3, 
                                     edgecolor='red', facecolor='red', alpha=0.25)
            ax.add_patch(rect)
        elif short_name == "M":
            # Grupo 1: E.~P -> Linhas EN=11,10 (r=2,3) e Colunas PC=00,01 (c=0,1)
            rect1 = patches.Rectangle((-0.42, 2-0.42), 1.84, 1.84, linewidth=3, 
                                      edgecolor='blue', facecolor='blue', alpha=0.20, label='E . ~P')
            # Grupo 2: E.C -> Linhas EN=11,10 (r=2,3) e Colunas PC=01,11 (c=1,2)
            rect2 = patches.Rectangle((1-0.38, 2-0.38), 1.76, 1.76, linewidth=3, 
                                      edgecolor='green', facecolor='green', alpha=0.20, label='E . C')
            ax.add_patch(rect1)
            ax.add_patch(rect2)
            ax.legend(loc='lower left', frameon=True)
        elif short_name == "A":
            # Grupo 1: ~E -> Linhas EN=00,01 (r=0,1) e Colunas 0..3
            rect1 = patches.Rectangle((-0.42, -0.42), 3.84, 1.84, linewidth=3, 
                                      edgecolor='red', facecolor='red', alpha=0.20, label='~E')
            # Grupo 2: ~N.P.~C -> r=0,3 e c=3
            rect2 = patches.Rectangle((3-0.38, -0.38), 0.76, 3.76, linewidth=3, 
                                      edgecolor='orange', facecolor='orange', alpha=0.20, label='~N . P . ~C')
            ax.add_patch(rect1)
            ax.add_patch(rect2)
            ax.legend(loc='upper right', frameon=True)
            
    plt.tight_layout()
    plt.show()

plot_karnaugh_maps()

## 5. Análise Comparativa de Custo de Hardware e Otimização

Avaliamos a redução quantitativa de complexidade em termos de:
* **Contagem de Literais:** Número total de ocorrências de variáveis diretas ou complementadas.
* **Portas Lógicas Equivalentes:** Quantidade de portas AND, OR e NOT necessárias em uma implementação em circuito combinacional discreto.

In [ ]:
# Cálculo de Literais e Portas
def count_literals_and_gates(canonical_minterms_count, min_expr):
    # Canônica: cada mintermo tem 4 literais
    canon_literals = canonical_minterms_count * 4
    canon_and_gates = canonical_minterms_count
    canon_or_gates = 1 if canonical_minterms_count > 1 else 0
    canon_total_gates = canon_and_gates + canon_or_gates
    
    # Mínima:
    terms = min_expr.split(" + ")
    min_literals = sum([len(t.split(" . ")) for t in terms if t not in ["0", "1"]])
    min_and_gates = len([t for t in terms if len(t.split(" . ")) > 1])
    min_or_gates = 1 if len(terms) > 1 else 0
    min_total_gates = min_and_gates + min_or_gates
    
    return canon_literals, min_literals, canon_total_gates, min_total_gates

comparison_rows = []
for short_name in ["V", "M", "A"]:
    m_count = len(canonical_results[short_name]["on_set"])
    min_expr = qm_results[short_name]["min_expr"]
    c_lit, m_lit, c_gate, m_gate = count_literals_and_gates(m_count, min_expr)
    
    saving_lit = ((c_lit - m_lit) / c_lit) * 100 if c_lit > 0 else 0
    saving_gate = ((c_gate - m_gate) / c_gate) * 100 if c_gate > 0 else 0
    
    comparison_rows.append({
        "Função": short_name,
        "Literais Canônicos": c_lit,
        "Literais Otimizados": m_lit,
        "Economia de Literais (%)": f"{saving_lit:.1f}%",
        "Portas Canônicas (AND/OR)": c_gate,
        "Portas Otimizadas (AND/OR)": m_gate,
        "Redução de Portas (%)": f"{saving_gate:.1f}%"
    })

df_comparison = pd.DataFrame(comparison_rows)
display(df_comparison)

# Gráfico de Barras Comparativo
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
x = np.arange(len(comparison_rows))
width = 0.35

# Literais
ax[0].bar(x - width/2, df_comparison["Literais Canônicos"], width, label='Canônica (FND)', color='#e74c3c')
ax[0].bar(x + width/2, df_comparison["Literais Otimizados"], width, label='Otimizada (QM)', color='#2ecc71')
ax[0].set_ylabel('Número de Literais', fontweight='bold')
ax[0].set_title('Comparativo de Literais (Complexidade Lógica)', fontweight='bold')
ax[0].set_xticks(x)
ax[0].set_xticklabels(df_comparison["Função"], fontweight='bold')
ax[0].legend()

# Portas
ax[1].bar(x - width/2, df_comparison["Portas Canônicas (AND/OR)"], width, label='Canônica (FND)', color='#e67e22')
ax[1].bar(x + width/2, df_comparison["Portas Otimizadas (AND/OR)"], width, label='Otimizada (QM)', color='#3498db')
ax[1].set_ylabel('Quantidade de Portas AND/OR', fontweight='bold')
ax[1].set_title('Comparativo de Portas Lógicas (Custo de Hardware)', fontweight='bold')
ax[1].set_xticks(x)
ax[1].set_xticklabels(df_comparison["Função"], fontweight='bold')
ax[1].legend()

plt.tight_layout()
plt.show()

## 6. Implementação Industrial e Simulação Dinâmica

Demonstramos a conversão direta das equações booleanas mínimas para **Texto Estruturado (IEC 61131-3 ST)** e simulamos um ciclo completo de operação da esteira e envase com 10 passos temporais.

In [ ]:
# Simulação de um ciclo temporal da estação de envase
# Cenário de teste:
# Passo 0..1: Esteira buscando garrafa (P=0)
# Passo 2: Garrafa chega sob o bico (P=1, C=0) -> Envase inicia (V=1, M=0)
# Passo 3..4: Envase continua (P=1, C=0)
# Passo 5: Sensor de nível da garrafa aciona (P=1, C=1) -> Envase encerra (V=0), esteira retoma (M=1)
# Passo 6: Garrafa deixa o posto (P=0, C=0)
# Passo 7: Tanque esvazia com nova garrafa (P=1, C=0, N=0) -> Alarme dispara (A=1)
# Passo 8..9: Botão de emergência acionado (E=0) -> Parada total de segurança

timeline_inputs = [
    {"t": "t0", "E": 1, "N": 1, "P": 0, "C": 0, "Evento": "Início do ciclo: esteira vazia"},
    {"t": "t1", "E": 1, "N": 1, "P": 0, "C": 0, "Evento": "Garrafa em trânsito"},
    {"t": "t2", "E": 1, "N": 1, "P": 1, "C": 0, "Evento": "Garrafa detectada no posto -> Envase inicia"},
    {"t": "t3", "E": 1, "N": 1, "P": 1, "C": 0, "Evento": "Líquido sendo dosado na garrafa"},
    {"t": "t4", "E": 1, "N": 1, "P": 1, "C": 0, "Evento": "Líquido atingindo nível nominal"},
    {"t": "t5", "E": 1, "N": 1, "P": 1, "C": 1, "Evento": "Garrafa cheia -> Válvula fecha, esteira retoma"},
    {"t": "t6", "E": 1, "N": 1, "P": 0, "C": 0, "Evento": "Garrafa sai do bico em direção ao tampador"},
    {"t": "t7", "E": 1, "N": 0, "P": 1, "C": 0, "Evento": "Tanque vazio com garrafa no posto -> FALHA/ALARME!"},
    {"t": "t8", "E": 0, "N": 0, "P": 1, "C": 0, "Evento": "Operador pressiona botão de EMERGÊNCIA"},
    {"t": "t9", "E": 0, "N": 1, "P": 0, "C": 0, "Evento": "Emergência mantida"}
]

sim_results = []
for row in timeline_inputs:
    E, N, P, C = row["E"], row["N"], row["P"], row["C"]
    
    # Avaliação pelas equações mínimas
    V = int(E and N and P and (not C))
    M = int(E and ((not P) or C))
    A = int((not E) or ((not N) and P and (not C)))
    
    sim_results.append({
        "Tempo": row["t"],
        "Evento / Cenário": row["Evento"],
        "E": E, "N": N, "P": P, "C": C,
        "Válvula (V)": V,
        "Motor (M)": M,
        "Alarme (A)": A
    })

df_sim = pd.DataFrame(sim_results)
display(df_sim)

# Plot da simulação temporal
fig, ax = plt.subplots(figsize=(14, 4.5))
time_steps = [r["Tempo"] for r in sim_results]
v_vals = [r["Válvula (V)"] for r in sim_results]
m_vals = [r["Motor (M)"] for r in sim_results]
a_vals = [r["Alarme (A)"] for r in sim_results]

x_pos = np.arange(len(time_steps))
ax.step(x_pos, np.array(v_vals) + 2.2, where='mid', label='Válvula de Envase (V)', color='blue', linewidth=2.5)
ax.step(x_pos, np.array(m_vals) + 1.1, where='mid', label='Motor da Esteira (M)', color='green', linewidth=2.5)
ax.step(x_pos, np.array(a_vals), where='mid', label='Alarme de Falha (A)', color='red', linewidth=2.5)

ax.set_yticks([0, 1, 1.1, 2.1, 2.2, 3.2])
ax.set_yticklabels(['0 (OFF)', '1 (ON)', '0 (OFF)', '1 (ON)', '0 (OFF)', '1 (ON)'])
ax.set_xticks(x_pos)
ax.set_xticklabels([f"{t}\n({r['E']}{r['N']}{r['P']}{r['C']})" for t, r in zip(time_steps, sim_results)], fontsize=9)
ax.set_title('Cronograma de Acionamento dos Atuadores durante a Simulação Temporal', fontweight='bold', fontsize=12)
ax.set_xlabel('Instantes de Tempo (Vetor de Entrada E N P C)', fontweight='bold')
ax.legend(loc='upper right', frameon=True)
ax.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

## 7. Conclusões e Considerações Finais

1. **Eficiência Algébrica e Computacional:** A minimização sistemática reduziu o custo lógico do motor e alarme em quase **$90\%$**, permitindo uma implementação direta, limpa e imune a condições de corrida (*glitches*).
2. **Integração com a Engenharia:** As equações mínimas obtidas alimentam diretamente a programação em **Diagrama Ladder (LD)** e **Texto Estruturado (ST)** em conformidade com as normas **IEC 61131-3** e **NR-12**.
3. **Próximas Etapas do Projeto:** Os resultados deste módulo servirão de base para a modelagem da máquina de estados sequencial (GRAFCET / SFC / FSM) da linha integrada de envasamento e tampamento de bebidas.